# Redraw route figures + tables from saved results

Pure-visualization notebook. **Reads only**: every artifact under
`artifacts/results/` -- both the route-payload `*_routes.pt` files and the
standalone `*.csv` tables produced by the experiment notebooks
(`evaluation_seeded_lc_bco_mumford0.ipynb`, `lc_improvement_training.ipynb`,
the benchmark / NSGA-II / MACSA sections, `rerun_nsgaii.ipynb`) is reloaded
and re-rendered here. No experiment is re-run, no `.pt` / `.csv` file is
overwritten.

For each `*_routes.pt` payload the notebook shows three things in order:

1. **Comparison table** -- built from each `RunResult.metrics` via the shared
   `build_comparison_table` (one row per run, cost / ATT / RTT / d_un / deltas
   vs the initial-kind row; numeric columns rounded to `TABLE_DECIMALS = 3`).
2. **Diff figure** -- cell 0 is the reference (first run, typically the
   initial / seed routes), cells 1.. are the route diffs vs the reference.
3. **Plain figure** -- companion view where every run is drawn as a plain
   route set, with `cost=...` in each subtitle.

Then a separate section dumps every standalone CSV in `artifacts/results/`
(comparison tables, summary tables, train-case scenarios, ...).

All metric numbers use the `:.2f` fixed-point format (no `e` notation) in
figure subtitles and `.round(3)` in tables, controlled by `_fmt` in
`eval_lib/figures.py` and `TABLE_DECIMALS` in `eval_lib/tables.py`.

## 1. Imports + discovery

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

from eval_lib import *  # noqa: F401,F403
from eval_lib import (load_route_results, render_route_comparison_figure,
                      render_route_set_figure, build_comparison_table,
                      RESULTS_DIR)

# Wide DataFrames render in full; rounded numbers stay readable.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

# Discover every saved route payload. Filenames end with `_routes.pt` -- strip
# that suffix to recover the experiment name accepted by `load_route_results`.
payload_paths = sorted(RESULTS_DIR.glob('*_routes.pt'))
experiment_names = [p.name[:-len('_routes.pt')] for p in payload_paths]
print(f'Found {len(experiment_names)} saved route payloads:')
for name in experiment_names:
    print(f'  {name}')

# Discover every standalone CSV (tables that are not coupled to a .pt).
csv_paths = sorted(RESULTS_DIR.glob('*.csv'))
print(f'\nFound {len(csv_paths)} standalone CSV tables:')
for path in csv_paths:
    print(f'  {path.name}')

Found 20 saved route payloads:
  adaptation_control
  adaptation_demand
  adaptation_edge
  benchmark_Mandl
  benchmark_Mumford0
  benchmark_Mumford1
  benchmark_Mumford2
  benchmark_Mumford3
  macsa_mandl_8_without_worse
  macsa_mandl_8_worse
  mumford0_lc_routes_without_worse
  mumford0_lc_routes_worse
  nx_dataset_routes_without_worse
  nx_dataset_routes_worse
  objective_weight_balanced_0.33_0.33_0.33
  objective_weight_demand-only_1_0_0
  objective_weight_route-only_0_1_0
  train_case_Best_positive_improvement_case
  train_case_Extreme_trim-heavy_stress_case
  train_case_Median_positive_improvement_case

Found 14 standalone CSV tables:
  baseline_optimizers_comparison.csv
  benchmark_sweep.csv
  macsa_comparison.csv
  macsa_summary.csv
  mumford0_lc_comparison.csv
  nsgaii_comparison.csv
  nx_dataset_comparison.csv
  nx_heuristic_dataset_comparison.csv
  objective_weight_comparison.csv
  train_case_Best_positive_improvement_case_scenarios.csv
  train_case_Extreme_trim-heavy_stress

## 2. Per-payload tables + figures

For each `*_routes.pt`: comparison table (from `RunResult.metrics`) + diff
figure + plain figure. Skips the diff figure when only one run is stored
(no reference / case split possible).

In [ ]:
for name in experiment_names:
    results, coords, street_adj = load_route_results(name)
    if not results:
        print(f'[skip] {name}: empty payload')
        continue

    print(f'\n=== {name} ===')

    # Comparison table built from the stored RunResult.metrics. Already
    # rounded to TABLE_DECIMALS; deltas vs the kind="initial" row are added
    # automatically when present. When the payload has both worse /
    # without_worse BCO runs, split into two compact tables; otherwise
    # display the single table as is.
    table = build_comparison_table(results)
    if table.empty:
        print(f'[note] {name}: build_comparison_table returned empty')
    else:
        if ('accept_mode' in table.columns and
                table['accept_mode'].dropna().nunique() > 1):
            _wo, _wr = split_comparison_table_by_accept_mode(table)
            print(f'   accept=without_worse')
            display(_wo)
            print(f'   accept=worse')
            display(_wr)
        else:
            display(table)

    # 1) Diff figure -- only meaningful when there are at least 2 runs.
    if len(results) >= 2:
        fig_diff = render_route_comparison_figure(
            results[0], results[1:], coords, street_adj,
            title=f'{name} (diff vs {results[0].label or "reference"})',
            ncols=3)
        plt.show()
        plt.close(fig_diff)
    else:
        print(f'[note] {name}: only one run stored, skipping diff figure')

    # 2) Plain figure -- every run drawn as a plain route set, cost in subtitle.
    fig_plain = render_route_set_figure(
        results, coords, street_adj,
        title=f'{name} (plain routes)',
        ncols=3)
    plt.show()
    plt.close(fig_plain)

## 3. Standalone CSV tables

Tables that live in `artifacts/results/` independently of any route payload
-- summary tables (`worse_accept_lc_summary`, `worse_accept_nx_dataset_summary`,
`macsa_summary`), per-experiment comparison tables
(`mumford0_lc_comparison`, `nx_dataset_comparison`,
`nx_heuristic_dataset_comparison`, `baseline_optimizers_comparison`,
`nsgaii_comparison`, `objective_weight_comparison`, `benchmark_sweep`,
`macsa_comparison`), and the train-case scenarios from
`lc_improvement_training.ipynb`.

Loaded via `pd.read_csv` and displayed verbatim -- they are saved already
rounded to `TABLE_DECIMALS = 3` by the library `save_table` /
`build_comparison_table` pipeline.

In [ ]:
for path in csv_paths:
    print(f'\n=== {path.name} ===')
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        print(f'  [load failed: {exc!r}]')
        continue
    if df.empty:
        print('  (empty)')
        continue
    print(f'  rows: {len(df)}, columns: {len(df.columns)}')
    display(df)